In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FINAL_DIR = PROJECT_ROOT / "data" / "final"
TABLES_DIR = PROJECT_ROOT / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    FINAL_DIR / "model_features_20d_provisional.csv",
    parse_dates=["Date"]
)

df = df.sort_values("Date").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

Dataset shape: (3944, 17)
Date range: 2010-03-31 00:00:00 to 2025-12-02 00:00:00


In [2]:
MARKET_FEATURES = [
    "Return_Lag1",
    "Absolute_Return",
    "RV_Historical_5D",
    "RV_Historical_10D",
    "RV_Historical_20D",
    "RV_Historical_60D",
    "RV_20D_Lag1"
]

GPR_FEATURES = [
    "GPR_Level",
    "GPR_Change",
    "GPR_MA5",
    "GPR_MA20"
]

EPU_FEATURES = [
    "EPU_Level",
    "EPU_Change",
    "EPU_MA5",
    "EPU_MA20"
]

FEATURE_SETS = {
    "A_Baseline": MARKET_FEATURES,
    "B_GPR": MARKET_FEATURES + GPR_FEATURES,
    "C_EPU": MARKET_FEATURES + EPU_FEATURES,
    "D_Combined": MARKET_FEATURES + GPR_FEATURES + EPU_FEATURES
}

TARGET = "RV_Future_20D"

for name, features in FEATURE_SETS.items():
    print(f"{name}: {len(features)} predictors")

A_Baseline: 7 predictors
B_GPR: 11 predictors
C_EPU: 11 predictors
D_Combined: 15 predictors


In [3]:
HORIZON = 20

validation_start = pd.Timestamp("2020-01-01")

first_validation_position = df.index[
    df["Date"] >= validation_start
][0]

# A training target at position i uses returns through i + 20.
# Require the complete target to be observed before validation begins.
last_train_position = first_validation_position - HORIZON - 1

train = df.iloc[:last_train_position + 1].copy()

validation = df[
    (df["Date"] >= "2020-01-01") &
    (df["Date"] < "2023-01-01")
].copy()

print("Training observations:", len(train))
print("Validation observations:", len(validation))

print("Last training date:", train["Date"].max())
print("First validation date:", validation["Date"].min())

assert len(train) == 2436
assert len(validation) == 756

assert train.index.max() + HORIZON < validation.index.min()

print("\nChronological split and purge checks passed!")

Training observations: 2436
Validation observations: 756
Last training date: 2019-12-02 00:00:00
First validation date: 2020-01-02 00:00:00

Chronological split and purge checks passed!


In [5]:
def evaluate_forecast(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(y_true, y_pred)

    epsilon = 1e-4

    safe_mape = np.mean(
        np.abs(y_true - y_pred) /
        np.maximum(np.abs(y_true), epsilon)
    ) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "Safe_MAPE": safe_mape
    }

In [6]:
X_train = train[MARKET_FEATURES]
y_train = train[TARGET]

X_validation = validation[MARKET_FEATURES]
y_validation = validation[TARGET]

rf_baseline = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train, y_train)

rf_predictions = rf_baseline.predict(X_validation)

rf_predictions = np.maximum(rf_predictions, 0)

rf_metrics = evaluate_forecast(
    y_validation,
    rf_predictions
)

print("=" * 60)
print("RANDOM FOREST — MARKET-ONLY VALIDATION")
print("=" * 60)

for metric, value in rf_metrics.items():
    print(f"{metric}: {value:.6f}")

RANDOM FOREST — MARKET-ONLY VALIDATION
RMSE: 0.038677
MAE: 0.019927
Safe_MAPE: 27.855195


In [7]:
comparison = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "RMSE": 0.035895,
        "MAE": 0.018616,
        "Safe_MAPE": 27.730350
    },
    {
        "Model": "Persistence",
        "RMSE": 0.043013,
        "MAE": 0.023945,
        "Safe_MAPE": 37.751981
    },
    {
        "Model": "Random Forest",
        **rf_metrics
    }
])

comparison = comparison.sort_values(
    "RMSE"
).reset_index(drop=True)

display(comparison.round(6))

comparison.to_csv(
    TABLES_DIR / "random_forest_baseline_comparison.csv",
    index=False
)

,Model,RMSE,MAE,Safe_MAPE
0,Linear Regression,0.035895,0.018616,27.730350
1,Random Forest,0.038677,0.019927,27.855195
2,Persistence,0.043013,0.023945,37.751981


In [8]:
rf_results = []

rf_models = {}

rf_predictions_by_set = {}

for experiment, features in FEATURE_SETS.items():

    print(f"\nTraining: {experiment}")

    X_train = train[features]
    y_train = train[TARGET]

    X_validation = validation[features]
    y_validation = validation[TARGET]

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)

    predictions = np.maximum(predictions, 0)

    metrics = evaluate_forecast(
        y_validation,
        predictions
    )

    rf_results.append({
        "Experiment": experiment,
        "Predictors": len(features),
        **metrics
    })

    rf_models[experiment] = model

    rf_predictions_by_set[experiment] = predictions

rf_results_df = pd.DataFrame(rf_results)

rf_results_df = rf_results_df.sort_values(
    "RMSE"
).reset_index(drop=True)

print("\n" + "=" * 60)
print("RANDOM FOREST FEATURE ABLATION RESULTS")
print("=" * 60)

display(rf_results_df.round(6))

rf_results_df.to_csv(
    TABLES_DIR / "random_forest_feature_ablation_provisional.csv",
    index=False
)


Training: A_Baseline

Training: B_GPR

Training: C_EPU

Training: D_Combined

RANDOM FOREST FEATURE ABLATION RESULTS


,Experiment,Predictors,RMSE,MAE,Safe_MAPE
0,B_GPR,11,0.038056,0.020190,29.726326
1,A_Baseline,7,0.038677,0.019927,27.855195
2,C_EPU,11,0.040895,0.021896,32.991910
3,D_Combined,15,0.041015,0.022758,34.952767


In [10]:
combined_model = rf_models["D_Combined"]

importance_df = pd.DataFrame({
    "Feature": FEATURE_SETS["D_Combined"],
    "Importance": combined_model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("=" * 60)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 60)

display(importance_df)

importance_df.to_csv(
    TABLES_DIR / "random_forest_feature_importance_provisional.csv",
    index=False
)

RANDOM FOREST FEATURE IMPORTANCE


,Feature,Importance
0,RV_Historical_10D,0.228238
1,RV_Historical_20D,0.137962
2,RV_Historical_60D,0.129880
3,GPR_MA20,0.108898
4,RV_Historical_5D,0.107030
5,EPU_MA20,0.094288
6,EPU_MA5,0.064533
7,RV_20D_Lag1,0.034875
8,GPR_MA5,0.026076
9,Return_Lag1,0.020763
